# TP1 - Ejercicio 2: Monitor de Jurassic Park


## Que pide el enunciado

- 5 zonas: Area de Velociraptores, Sector del Tiranosaurio, Recinto de los Triceratops, Centro de Visitantes, Laboratorio Genetico.
- Cada zona tiene un sistema de vigilancia **independiente y concurrente** (uno por zona).
- Cada cierto intervalo (parametro de **frecuencia**), cada sistema genera un evento al azar segun las probabilidades propias de su zona y lo imprime como `[ZONA] - EVENTO`.
- Esto se repite durante un tiempo total (parametro de **duracion**).
- Al terminar, cada sistema informa: la zona que vigilaba, el total de eventos detectados y el total de eventos **criticos** (dinosaurio fuera de su recinto, falla en el cerco electrico, perdida de comunicacion, alerta de seguridad).

## Como se resuelve

Se usa `multiprocessing.Process` (no `threading.Thread`): crea procesos reales del sistema operativo, coherente con "Procesos Pesados".

- `ZONES` define los eventos posibles de cada zona con su probabilidad y si es o no critico.
- `monitor_zone` es el codigo que corre cada proceso hijo: en un bucle, espera el intervalo, elige un evento al azar respetando las probabilidades (`random.choices` con pesos), lo imprime, cuenta totales, y al final imprime el resumen de esa zona.
- En `main`, se crean los 5 procesos y se arrancan **todos** con `.start()` en un primer `for` (creacion concurrente); recien en un segundo `for` se los espera con `.join()`.
- Cada proceso hace `random.seed()` al arrancar: se crea los procesos por defecto con `fork()`, que copia el estado del generador aleatorio del padre; sin este reseed, las 5 zonas podrian repetir la misma secuencia de eventos.

## Programa

In [1]:
%%writefile jurassic_park_monitor.py

import argparse
import os
import random
import time
from multiprocessing import Process


ZONES = {
    "Sector del Tiranosaurio": [
        ("Todo normal", 0.80, False),
        ("Tiranosaurio fuera del recinto", 0.10, True),
        ("Falla en el cerco electrico", 0.10, True),
    ],
    "Area de Velociraptores": [
        ("Todo normal", 0.70, False),
        ("Perdida de visibilidad", 0.20, False),
        ("Falla en el cerco electrico", 0.10, True),
    ],
    "Recinto de los Triceratops": [
        ("Todo normal", 0.60, False),
        ("Comportamiento inusual", 0.30, False),
        ("Estampida", 0.10, False),
    ],
    "Centro de Visitantes": [
        ("Todo normal", 0.80, False),
        ("Perdida de comunicacion", 0.15, True),
        ("Alerta de seguridad", 0.05, True),
    ],
    "Laboratorio Genetico": [
        ("Todo normal", 0.80, False),
        ("Falla del sistema", 0.10, False),
        ("Perdida de comunicacion", 0.05, True),
        ("Acceso no autorizado", 0.05, False),
    ],
}


def monitor_zone(zone_name, events, duration, interval):
    random.seed()  # cada proceso necesita su propia semilla de azar

    print(f"[{zone_name}] proceso iniciado (PID={os.getpid()})", flush=True)

    names = [e[0] for e in events]
    weights = [e[1] for e in events]
    is_critical = {e[0]: e[2] for e in events}

    total_events = 0
    total_critical = 0
    start_time = time.time()

    while time.time() - start_time < duration:
        time.sleep(interval)
        event = random.choices(names, weights=weights, k=1)[0]
        total_events += 1
        if is_critical[event]:
            total_critical += 1
        print(f"[{zone_name}] - {event}", flush=True)

    print(
        f"[{zone_name}] >> Monitoreo finalizado | "
        f"Total de eventos: {total_events} | Eventos criticos: {total_critical}",
        flush=True,
    )


def main():
    parser = argparse.ArgumentParser(description="Monitor concurrente de zonas de Jurassic Park")
    parser.add_argument("-d", "--duration", type=float, default=15.0,
                         help="Duracion total del monitoreo, en segundos")
    parser.add_argument("-i", "--interval", type=float, default=2.0,
                         help="Segundos entre reportes de cada zona")
    args = parser.parse_args()

    processes = [
        Process(target=monitor_zone, args=(zone, events, args.duration, args.interval))
        for zone, events in ZONES.items()
    ]


    for p in processes:
        p.start()

    for p in processes:
        p.join()

    print("Todos los sistemas de vigilancia finalizaron.")


if __name__ == "__main__":
    main()


Writing jurassic_park_monitor.py


## Ejecución

Corre por 12 segundos, reportando cada 2 segundos.

In [2]:
!python3 jurassic_park_monitor.py --duration 12 --interval 2

[Area de Velociraptores] proceso iniciado (PID=4748)
[Sector del Tiranosaurio] proceso iniciado (PID=4747)
[Recinto de los Triceratops] proceso iniciado (PID=4749)
[Centro de Visitantes] proceso iniciado (PID=4750)
[Laboratorio Genetico] proceso iniciado (PID=4751)
[Sector del Tiranosaurio] - Todo normal
[Area de Velociraptores] - Todo normal
[Recinto de los Triceratops] - Comportamiento inusual
[Laboratorio Genetico] - Todo normal
[Centro de Visitantes] - Todo normal
[Sector del Tiranosaurio] - Todo normal
[Area de Velociraptores] - Todo normal
[Recinto de los Triceratops] - Todo normal
[Laboratorio Genetico] - Falla del sistema
[Centro de Visitantes] - Todo normal
[Sector del Tiranosaurio] - Tiranosaurio fuera del recinto
[Area de Velociraptores] - Perdida de visibilidad
[Recinto de los Triceratops] - Todo normal
[Laboratorio Genetico] - Todo normal
[Centro de Visitantes] - Todo normal
[Sector del Tiranosaurio] - Todo normal
[Area de Velociraptores] - Todo normal
[Recinto de los Tric